# Mathematical Engineering - Financial Engineering, FY 2025-2026
# Buy Side - Exercise 3: Covariance Estimation Techniques & Statistical Factor Models


In [1]:
# Importing the libraries
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from utilities.backtest import backtest
from utilities.covariance_utilities import prepare_rolling_estimation_window
from utilities.portfolio_optimization import (
    mean_variance_portfolio,
    minimum_variance_portfolio,
)
from utilities.principal_component_analysis import (
    principal_component_analysis,
    align_eigenvectors_to_previous,
    pca_denoise_covariance,
)
from utilities.shrinkage import (
    constant_corr_shrinkage,
    market_factor_shrinkage,
)


In [ ]:
# Reading the data
data_path = Path("..")  # !!! COMPLETE AS APPROPRIATE !!!

last_prices = pd.read_csv(
    data_path / "sx5e_underlyings.csv", index_col="Date", parse_dates=True
)

In [ ]:
# Compute daily returns
performance = last_prices.pct_change().iloc[1:]

print(f"Dataset period: {performance.index[0].date()} to {performance.index[-1].date()}")
print(f"Number of observations: {len(performance)}")
print(f"Number of assets: {performance.shape[1]}")

In [ ]:
# Parameters
time_horizon = 3 * 252  # 2 years of data for estimation
# Parameter determining the minimum coverage of returns required for an asset to be included
# in the optimization process
min_coverage = 0.95

In [ ]:
# Get the last actual day in the data for each month (monthly rebalancing)
rebalance_dates = pd.DatetimeIndex(
    performance.groupby(pd.Grouper(freq="ME"))
    .apply(lambda x: x.index[-1] if len(x) > 0 else None)
    .dropna()
    .values
)

print(f"Number of rebalance dates: {len(rebalance_dates)}")
print(f"First rebalance: {rebalance_dates[0].date()}")
print(f"Last rebalance: {rebalance_dates[-1].date()}")

# Part I: Covariance Estimation

The sample covariance matrix is an unbiased estimator but can be noisy, especially when the number of observations $T$ is not much larger than the number of assets $N$. Shrinkage techniques improve estimation by combining the sample covariance with a structured target.

The shrunk covariance is computed as:
$$\hat{\Sigma}_{shrunk} = \delta \cdot F + (1 - \delta) \cdot S$$

where:
- $S$ is the sample covariance matrix
- $F$ is the structured target
- $\delta$ is the optimal shrinkage intensity (derived analytically)

a) Risk aversion parameter ($\gamma$) calibration.

In [ ]:
mean_variance_risk_aversion = 1  # !!! COMPLETE AS APPROPRIATE !!!

calibration_date = None
for date in rebalance_dates:
    candidate_window = prepare_rolling_estimation_window(
        returns=performance,
        rebalance_date=date,
        lookback=time_horizon,
        min_coverage=min_coverage,
    )
    if candidate_window.shape[0] == time_horizon and candidate_window.shape[1] > 0:
        calibration_date = date
        calibration_window = candidate_window
        break

if calibration_date is None:
    raise RuntimeError("No full estimation window available for portfolio checks")

calibration_sample_cov = calibration_window.cov()
calibration_expected_returns = calibration_window.mean(axis=0).values
ones = np.ones(calibration_sample_cov.shape[0])

calibration_mvp_weights = minimum_variance_portfolio(calibration_sample_cov.values)
calibration_mv_weights = mean_variance_portfolio(
    expected_returns=calibration_expected_returns,
    cov_matrix=calibration_sample_cov.values,
    risk_aversion=mean_variance_risk_aversion,
)

calibration_mvp_variance = float(
    calibration_mvp_weights @ calibration_sample_cov.values @ calibration_mvp_weights
)

np.testing.assert_allclose(calibration_mvp_weights.sum(), 1.0)
np.testing.assert_allclose(calibration_mv_weights.sum(), 1.0)

print(
    f"Closed-form checks on the {calibration_date.date()} rebalance window "
    f"({calibration_window.shape[1]} assets)"
)
pd.DataFrame(
    {
        "sum_weights": [calibration_mvp_weights.sum(), calibration_mv_weights.sum()],
        "in_sample_mean_return": [
            float(calibration_expected_returns @ calibration_mvp_weights),
            float(calibration_expected_returns @ calibration_mv_weights),
        ],
        "in_sample_volatility": [
            np.sqrt(calibration_mvp_variance),
            np.sqrt(
                float(
                    calibration_mv_weights
                    @ calibration_sample_cov.values
                    @ calibration_mv_weights
                )
            ),
        ],
        "gross_exposure": [
            np.abs(calibration_mvp_weights).sum(),
            np.abs(calibration_mv_weights).sum(),
        ],
    },
    index=["Minimum Variance", "Mean-Variance"],
)


b-d) Shrinkage, minimum variance and mean variance portfolios

In [ ]:
constant_corr_results = {}
mkt_factor_results = {}
sample_cov_min_var_ptfs = {}
constant_corr_cov_min_var_ptfs = {}
mkt_factor_cov_min_var_ptfs = {}
sample_cov_mean_var_ptfs = {}
constant_corr_cov_mean_var_ptfs = {}
mkt_factor_cov_mean_var_ptfs = {}
window_diagnostics = {}

for rebalance_date in rebalance_dates:
    cur_performance, cur_window_diagnostics = prepare_rolling_estimation_window(
        returns=performance,
        rebalance_date=rebalance_date,
        lookback=time_horizon,
        min_coverage=min_coverage,
        return_diagnostics=True,
    )
    if (
        cur_window_diagnostics["row_count"] < time_horizon
        or cur_performance.shape[1] == 0
    ):
        continue

    print(
        f"Processing: {rebalance_date.date()} ({cur_performance.shape[1]} assets kept)"
    )
    reb_date = rebalance_date.to_pydatetime().date()

    # Market returns as equal-weighted average
    market_returns = None # !!! COMPLETE AS APPROPRIATE !!!
    estimated_expected_returns = None # !!! COMPLETE AS APPROPRIATE !!!

    # Compute shrinkage estimators
    cur_constant_corr_results = constant_corr_shrinkage(returns=cur_performance)
    cur_mkt_factor_results = market_factor_shrinkage(
        returns=cur_performance, market_returns=market_returns
    )

    # Compute closed-form portfolios for each covariance estimator
    sample_cov = cur_constant_corr_results["sample_cov"]
    constant_corr_cov = cur_constant_corr_results["shrunk_cov"]
    mkt_factor_cov = cur_mkt_factor_results["shrunk_cov"]

    for cov, cov_min_var_ptfs, cov_mean_var_ptfs in zip(
        [sample_cov, constant_corr_cov, mkt_factor_cov],
        [
            sample_cov_min_var_ptfs,
            constant_corr_cov_min_var_ptfs,
            mkt_factor_cov_min_var_ptfs,
        ],
        [
            sample_cov_mean_var_ptfs,
            constant_corr_cov_mean_var_ptfs,
            mkt_factor_cov_mean_var_ptfs,
        ],
    ):
        aligned_expected_returns = estimated_expected_returns.values
        cov_min_var_ptfs[reb_date] = pd.Series(
            data=minimum_variance_portfolio(cov.values), index=cov.index
        )
        cov_mean_var_ptfs[reb_date] = pd.Series(
            data=mean_variance_portfolio(
                expected_returns=aligned_expected_returns,
                cov_matrix=cov.values,
                risk_aversion=mean_variance_risk_aversion,
            ),
            index=cov.index,
        )

    constant_corr_results[reb_date] = cur_constant_corr_results
    mkt_factor_results[reb_date] = cur_mkt_factor_results
    window_diagnostics[reb_date] = cur_window_diagnostics


d) Minimum variance portfolios out of sample evaluation

In [8]:
constant_corr_cov_min_var_ptf = pd.DataFrame(constant_corr_cov_min_var_ptfs).T.fillna(
    0.0
)
mkt_factor_cov_min_var_ptf = pd.DataFrame(mkt_factor_cov_min_var_ptfs).T.fillna(0.0)
sample_cov_min_var_ptf = pd.DataFrame(sample_cov_min_var_ptfs).T.fillna(0.0)


In [9]:
# Backtest portfolios
sample_cov_min_var_ptf_pfm = backtest(
    portfolios=sample_cov_min_var_ptf,
    returns=performance,
)
constant_corr_cov_min_var_ptf_pfm = backtest(
    portfolios=constant_corr_cov_min_var_ptf,
    returns=performance,
)
mkt_factor_cov_min_var_ptf_pfm = backtest(
    portfolios=mkt_factor_cov_min_var_ptf,
    returns=performance,
)

In [ ]:
# Plot rolling annualized volatility
plt.figure(figsize=(14, 5))

(
    np.sqrt(252) * sample_cov_min_var_ptf_pfm.pct_change().ewm(span=time_horizon).std()
).plot(label="Sample Covariance", linewidth=1.5)

(
    np.sqrt(252)
    * constant_corr_cov_min_var_ptf_pfm.pct_change().ewm(span=time_horizon).std()
).plot(label="Constant Correlation Shrinkage", linewidth=1.5)

(
    np.sqrt(252)
    * mkt_factor_cov_min_var_ptf_pfm.pct_change().ewm(span=time_horizon).std()
).plot(label="Market Factor Shrinkage", linewidth=1.5)

plt.xlabel("Date")
plt.ylabel("Annualized Volatility")
plt.title("Rolling Annualized Volatility of Minimum Variance Portfolios")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot cumulative returns
plt.figure(figsize=(14, 5))

sample_cov_min_var_ptf_pfm.plot(label="Sample Covariance", linewidth=1.5)
constant_corr_cov_min_var_ptf_pfm.plot(
    label="Constant Correlation Shrinkage", linewidth=1.5
)
mkt_factor_cov_min_var_ptf_pfm.plot(label="Market Factor Shrinkage", linewidth=1.5)

plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.title("Cumulative Returns of Minimum Variance Portfolios")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot gross exposure
plt.figure(figsize=(14, 5))

sample_cov_min_var_ptf.diff().abs().sum(axis=1).plot(
    label="Sample Covariance", linewidth=1.5
)
constant_corr_cov_min_var_ptf.diff().abs().sum(axis=1).plot(
    label="Constant Correlation Shrinkage", linewidth=1.5
)
mkt_factor_cov_min_var_ptf.diff().abs().sum(axis=1).plot(
    label="Market Factor Shrinkage", linewidth=1.5
)

plt.xlabel("Date")
plt.ylabel("Turnover")
plt.title("Turnover of Minimum Variance Portfolios")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Print turnover statistics
print("Average Turnover:")
print(
    f"  Sample Covariance: {sample_cov_min_var_ptf.diff().abs().sum(axis=1).mean():.2f}"
)
print(
    f"  Constant Corr Shrinkage: {constant_corr_cov_min_var_ptf.diff().abs().sum(axis=1).mean():.2f}"
)
print(
    f"  Market Factor Shrinkage: {mkt_factor_cov_min_var_ptf.diff().abs().sum(axis=1).mean():.2f}"
)

In [ ]:
# Plot gross exposure
plt.figure(figsize=(14, 5))

sample_cov_min_var_ptf.abs().sum(axis=1).plot(label="Sample Covariance", linewidth=1.5)
constant_corr_cov_min_var_ptf.abs().sum(axis=1).plot(
    label="Constant Correlation Shrinkage", linewidth=1.5
)
mkt_factor_cov_min_var_ptf.abs().sum(axis=1).plot(
    label="Market Factor Shrinkage", linewidth=1.5
)

plt.xlabel("Date")
plt.ylabel("Gross Exposure")
plt.title("Gross Exposure of Minimum Variance Portfolios")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Print exposure statistics
print("Average Gross Exposure:")
print(f"  Sample Covariance: {sample_cov_min_var_ptf.abs().sum(axis=1).mean():.2f}")
print(
    f"  Constant Corr Shrinkage: {constant_corr_cov_min_var_ptf.abs().sum(axis=1).mean():.2f}"
)
print(
    f"  Market Factor Shrinkage: {mkt_factor_cov_min_var_ptf.abs().sum(axis=1).mean():.2f}"
)

d) Mean-variance portfolios out of sample evaluation


In [14]:
constant_corr_cov_mean_var_ptf = pd.DataFrame(constant_corr_cov_mean_var_ptfs).T.fillna(
    0.0
)
mkt_factor_cov_mean_var_ptf = pd.DataFrame(mkt_factor_cov_mean_var_ptfs).T.fillna(0.0)
sample_cov_mean_var_ptf = pd.DataFrame(sample_cov_mean_var_ptfs).T.fillna(0.0)


In [15]:
# Backtest portfolios
sample_cov_mean_var_ptf_pfm = backtest(
    portfolios=sample_cov_mean_var_ptf,
    returns=performance,
)
constant_corr_cov_mean_var_ptf_pfm = backtest(
    portfolios=constant_corr_cov_mean_var_ptf,
    returns=performance,
)
mkt_factor_cov_mean_var_ptf_pfm = backtest(
    portfolios=mkt_factor_cov_mean_var_ptf,
    returns=performance,
)


In [ ]:
# Plot rolling annualized volatility
plt.figure(figsize=(14, 5))

(
    np.sqrt(252) * sample_cov_mean_var_ptf_pfm.pct_change().ewm(span=time_horizon).std()
).plot(label="Sample Covariance", linewidth=1.5)

(
    np.sqrt(252)
    * constant_corr_cov_mean_var_ptf_pfm.pct_change().ewm(span=time_horizon).std()
).plot(label="Constant Correlation Shrinkage", linewidth=1.5)

(
    np.sqrt(252)
    * mkt_factor_cov_mean_var_ptf_pfm.pct_change().ewm(span=time_horizon).std()
).plot(label="Market Factor Shrinkage", linewidth=1.5)

plt.xlabel("Date")
plt.ylabel("Annualized Volatility")
plt.title("Rolling Annualized Volatility of Mean-Variance Portfolios")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Plot cumulative returns
plt.figure(figsize=(14, 5))

sample_cov_mean_var_ptf_pfm.plot(label="Sample Covariance", linewidth=1.5)
constant_corr_cov_mean_var_ptf_pfm.plot(
    label="Constant Correlation Shrinkage", linewidth=1.5
)
mkt_factor_cov_mean_var_ptf_pfm.plot(label="Market Factor Shrinkage", linewidth=1.5)

plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.title("Cumulative Returns of Mean-Variance Portfolios")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Plot gross exposure
plt.figure(figsize=(14, 5))

sample_cov_mean_var_ptf.diff().abs().sum(axis=1).plot(
    label="Sample Covariance", linewidth=1.5
)
constant_corr_cov_mean_var_ptf.diff().abs().sum(axis=1).plot(
    label="Constant Correlation Shrinkage", linewidth=1.5
)
mkt_factor_cov_mean_var_ptf.diff().abs().sum(axis=1).plot(
    label="Market Factor Shrinkage", linewidth=1.5
)

plt.xlabel("Date")
plt.ylabel("Turnover")
plt.title("Turnover of Mean-Variance Portfolios")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Print turnover statistics
print("Average Turnover:")
print(
    f"  Sample Covariance: {sample_cov_mean_var_ptf.diff().abs().sum(axis=1).mean():.2f}"
)
print(
    f"  Constant Corr Shrinkage: {constant_corr_cov_mean_var_ptf.diff().abs().sum(axis=1).mean():.2f}"
)
print(
    f"  Market Factor Shrinkage: {mkt_factor_cov_mean_var_ptf.diff().abs().sum(axis=1).mean():.2f}"
)

In [ ]:
# Plot gross exposure
plt.figure(figsize=(14, 5))

sample_cov_mean_var_ptf.abs().sum(axis=1).plot(label="Sample Covariance", linewidth=1.5)
constant_corr_cov_mean_var_ptf.abs().sum(axis=1).plot(
    label="Constant Correlation Shrinkage", linewidth=1.5
)
mkt_factor_cov_mean_var_ptf.abs().sum(axis=1).plot(
    label="Market Factor Shrinkage", linewidth=1.5
)

plt.xlabel("Date")
plt.ylabel("Gross Exposure")
plt.title("Gross Exposure of Mean-Variance Portfolios")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Print exposure statistics
print("Average Gross Exposure:")
print(f"  Sample Covariance: {sample_cov_mean_var_ptf.abs().sum(axis=1).mean():.2f}")
print(
    f"  Constant Corr Shrinkage: {constant_corr_cov_mean_var_ptf.abs().sum(axis=1).mean():.2f}"
)
print(
    f"  Market Factor Shrinkage: {mkt_factor_cov_mean_var_ptf.abs().sum(axis=1).mean():.2f}"
)


Shrinkage intensity

In [ ]:
# Plot shrinkage intensity over time
plt.figure(figsize=(14, 5))

pd.Series(
    {date: results["intensity"] for date, results in constant_corr_results.items()}
).plot(label="Constant Correlation Shrinkage", linewidth=1.5)

pd.Series(
    {date: results["intensity"] for date, results in mkt_factor_results.items()}
).plot(label="Market Factor Shrinkage", linewidth=1.5)

plt.xlabel("Date")
plt.ylabel("Shrinkage Intensity (δ)")
plt.title("Optimal Shrinkage Intensity Over Time")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Print statistics
const_corr_intensity = pd.Series(
    {date: results["intensity"] for date, results in constant_corr_results.items()}
)
mkt_factor_intensity = pd.Series(
    {date: results["intensity"] for date, results in mkt_factor_results.items()}
)

print("Constant Correlation Shrinkage Intensity:")
print(f"  Mean: {const_corr_intensity.mean():.2%}")
print(f"  Std: {const_corr_intensity.std():.2%}")
print("\nMarket Factor Shrinkage Intensity:")
print(f"  Mean: {mkt_factor_intensity.mean():.2%}")
print(f"  Std: {mkt_factor_intensity.std():.2%}")

Condition Number


In [ ]:
sample_cov_cond = {
    date: None  # !!! COMPLETE AS APPROPRIATE !!!
    for date, results in constant_corr_results.items()
}
constant_corr_cond = {
    date: None  # !!! COMPLETE AS APPROPRIATE !!!
    for date, results in constant_corr_results.items()
}
mkt_factor_cond = {
    date: None  # !!! COMPLETE AS APPROPRIATE !!!
    for date, results in mkt_factor_results.items()
}
eligible_asset_counts = pd.Series(
    {
        date: diagnostics["asset_count_after_filter"]
        for date, diagnostics in window_diagnostics.items()
    }
)


In [ ]:
# Plot condition numbers
plt.figure(figsize=(14, 5))

pd.Series(sample_cov_cond).plot(label="Sample Covariance", linewidth=1.5)
pd.Series(constant_corr_cond).plot(
    label="Constant Correlation Shrinkage", linewidth=1.5
)
pd.Series(mkt_factor_cond).plot(label="Market Factor Shrinkage", linewidth=1.5)

plt.xlabel("Date")
plt.ylabel("Condition Number")
plt.yscale("log")
plt.title("Condition Number of Covariance Matrices Over Time (log scale)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Print statistics
print("Condition Number Statistics:")
print(f"  Sample Covariance - Mean: {np.mean(list(sample_cov_cond.values())):.1f}")
print(
    f"  Constant Corr Shrinkage - Mean: {np.mean(list(constant_corr_cond.values())):.1f}"
)
print(
    f"  Market Factor Shrinkage - Mean: {np.mean(list(mkt_factor_cond.values())):.1f}"
)
print(f"  Average number of eligible assets: {eligible_asset_counts.mean():.1f}")


# Part II: Statistical Factor Model

Principal Component Analysis (PCA) extracts the dominant risk factors from the correlation/covariance matrix. The first principal component typically represents the market factor, explaining a large portion of total variance.

Key properties of PCA:
- Eigenvalues represent variance explained by each component
- Eigenvectors represent factor loadings (exposures of each asset to the factor)
- Components are orthogonal (uncorrelated)

a) PCA

In [ ]:
eig_vals = {}
eig_vecs = {}
cosine_distances = {}
max_cosine_distances = {}

prev_eig_vecs = None
for rebalance_date in rebalance_dates:
    cur_performance = prepare_rolling_estimation_window(
        returns=performance,
        rebalance_date=rebalance_date,
        lookback=time_horizon,
        min_coverage=min_coverage,
    )
    if cur_performance.shape[0] < time_horizon or cur_performance.shape[1] == 0:
        continue

    print(
        f"Processing: {rebalance_date.date()} ({cur_performance.shape[1]} assets kept)"
    )
    reb_date = rebalance_date.to_pydatetime().date()

    # Compute covariance matrix and PCA
    sample_cov = cur_performance.cov()
    cur_eig_vals, cur_eig_vecs = principal_component_analysis(sample_cov.values)

    cur_eig_vals = pd.Series(
        data=cur_eig_vals.flatten(), index=range(len(cur_eig_vals))
    )
    cur_eig_vecs = pd.DataFrame(
        data=cur_eig_vecs,
        index=cur_performance.columns,
        columns=range(cur_eig_vecs.shape[1]),
    )
    cur_eig_vecs = align_eigenvectors_to_previous(cur_eig_vecs, prev_eig_vecs)

    # Compute cosine similarity between consecutive periods after sign alignment
    if prev_eig_vecs is not None:
        common_assets = cur_eig_vecs.index.intersection(prev_eig_vecs.index)
        components_num = min(
            len(common_assets),
            cur_eig_vecs.shape[1],
            prev_eig_vecs.shape[1],
        )

        a = cur_eig_vecs.loc[common_assets].iloc[:, :components_num]
        b = prev_eig_vecs.loc[common_assets].iloc[:, :components_num]
        pc_proximity = None  # !!! COMPLETE AS APPROPRIATE !!!

        cosine_distances[reb_date] = pd.Series(
            data=np.diag(pc_proximity),
            index=pc_proximity.index,
        )
        max_cosine_distances[reb_date] = pd.Series(
            data=pc_proximity.abs().max(axis=1).values.flatten(),
            index=pc_proximity.index,
        )

    prev_eig_vecs = cur_eig_vecs
    eig_vals[reb_date] = cur_eig_vals
    eig_vecs[reb_date] = cur_eig_vecs


b) Results discussion

Cosine similarity evolution after sign alignment

In [ ]:
# Plot cosine similarity evolution for the first few PCs
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, pc in enumerate([0, 1, 2, 3]):
    ax = axes[idx // 2, idx % 2]

    cosine_series = pd.Series(
        {
            date: cosine_distance[pc]
            for date, cosine_distance in cosine_distances.items()
        }
    )
    cosine_series.plot(ax=ax, linewidth=1.5)

    ax.set_xlabel("Date")
    ax.set_ylabel("Cosine Similarity")
    ax.set_title(f"Cosine Similarity Evolution for PC{pc + 1}")
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3)
    ax.axhline(y=0.9, color="r", linestyle="--", alpha=0.5, label="0.9 threshold")
    ax.legend()

plt.tight_layout()
plt.show()


Explained variance evolution

In [ ]:
# Plot the proportion of PCs needed to explain a given variance threshold
var_threshold = 0.75

plt.figure(figsize=(14, 5))

pc_proportion = pd.Series(
    {
        date: None  # !!! COMPLETE AS APPROPRIATE !!!
        for date, cur_eig_vals in eig_vals.items()
    }
)
pc_proportion.plot(linewidth=1.5)

plt.xlabel("Date")
plt.ylabel("Proportion of PCs")
plt.title(f"Proportion of PCs Needed to Explain {var_threshold * 100:.0f}% of Variance")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Statistics:")
print(f"  Mean proportion: {pc_proportion.mean():.2%}")
print(f"  Min proportion: {pc_proportion.min():.2%}")
print(f"  Max proportion: {pc_proportion.max():.2%}")

PCs interpretation

The factor loadings (eigenvector components) can be interpreted to understand what each PC represents. Typically:
- **PC1**: Market factor (similar loadings across all assets)
- **PC2+**: Sector/style factors (long-short patterns)

In [ ]:
# !!! COMPLETE AS APPROPRIATE !!!

# Part III: Connecting the Dots

Shrinkage and PCA-based denoising both address the same problem — the sample covariance matrix is noisy when $T/N$ is not large. They do so in complementary ways:

- **Shrinkage** blends the sample covariance with a structured target, pulling extreme (noisy) eigenvalues toward a common level.
- **PCA denoising** keeps the $k$ largest eigenvalues intact (signal) and replaces the remaining $N - k$ eigenvalues with their average (noise floor), then reconstructs: $$\hat{\Sigma}_{denoised} = \sum_{i=1}^{k} \lambda_i \mathbf{v}_i \mathbf{v}_i^\top + \bar{\lambda}_{noise} \sum_{i=k+1}^{N} \mathbf{v}_i \mathbf{v}_i^\top$$

a) PCA-based denoising of the covariance matrix

In [ ]:
# !!! COMPLETE AS APPROPRIATE !!!